<a href="https://colab.research.google.com/github/ParrvLuthra22/Drishti/blob/main/notebooks/phase4_videomae_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4 — VideoMAE Fight Detection Fine-tuning
- Model: MCG-NJU/videomae-base
- Dataset: RWF-2000 (1600 train, 400 val)
- Device: T4 GPU (Colab)


In [1]:
# Setup
!pip install torch torchvision transformers datasets scikit-learn supervision ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.5/95.5 kB 11.9 MB/s eta 0:00:00


In [2]:
# Clone repo
import sys, os

!git clone https://github.com/ParrvLuthra22/drishti.git
%cd drishti
sys.path.insert(0, '/content/drishti/src')
print("Repo ready")

Cloning into 'drishti'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 105 (delta 38), reused 81 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 95.89 KiB | 8.72 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/drishti
Repo ready


In [9]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip install kaggle -q

# Download RWF-2000 directly from Kaggle
!kaggle datasets download -d "vulamnguyen/rwf2000" \
    -p /content/drishti/data/ --unzip

# Check folder structure
!ls /content/drishti/data/
!ls /content/drishti/data/RWF-2000/ 2>/dev/null || \
 ls /content/drishti/data/rwf2000/ 2>/dev/null || \
 echo "check folder name above"


Dataset URL: https://www.kaggle.com/datasets/vulamnguyen/rwf2000
License(s): unknown
Resuming from 838860800 bytes (11478181001 bytes left)...
100% 11.5G/11.5G [09:00<00:00, 21.3MB/s]

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/kaggle/api/kaggle_api_extended.py", line 2476, in dataset_download_files
    z.extractall(effective_path)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/zipfile/__init__.py", line 1807, in extractall
    self._extract_member(zipinfo, path, pwd)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/zipfile/__init__.py", line 1878, in _extract_member
    open(targetpath, "wb") as target:
    ~~~~^^^^^^^^^^^^^^^^^^
OSError: [Errno 36] File name too long: '/content/drishti/data/RWF-2000/train/Fight/1060314╤äΓòò╨╜╤à╨│╨▓╤à╨┤╨╣╤å╨⌐╨»╤ë╨û╨╗╤ë╨⌐╨▓╤å╨É╨╡╤ê╨╕Γòæ╤à╨╛╨┤╤å╨¬Γöñ╤à╨Ü╨½╤äΓòæ╨¢╤äΓòùΓòó╤ÅΓò¥╨ÿ╤à╨¥╨«╤ë╨ù╨í╤àΓöéΓûæ╤ÅΓò¥╨Ö_17.avi'

During handling of the above exception, another exception occurr

In [10]:
import zipfile
import os

zip_path = '/content/drishti/data/rwf2000.zip'
extract_path = '/content/drishti/data/'

print("Extracting with error handling...")
skipped = []

with zipfile.ZipFile(zip_path, 'r') as z:
    for member in z.infolist():
        try:
            # Truncate filename if too long
            filename = member.filename
            parts = filename.split('/')
            if len(parts[-1]) > 200:
                ext = parts[-1].split('.')[-1]
                parts[-1] = parts[-1][:200] + '.' + ext
                member.filename = '/'.join(parts)
            z.extract(member, extract_path)
        except Exception as e:
            skipped.append(member.filename)

print(f"Done. Skipped {len(skipped)} files.")
print("\nFolder structure:")
!ls /content/drishti/data/RWF-2000/
!ls /content/drishti/data/RWF-2000/train/
!echo "Train Fight count:" && ls /content/drishti/data/RWF-2000/train/Fight/ | wc -l
!echo "Train NonFight count:" && ls /content/drishti/data/RWF-2000/train/NonFight/ | wc -l
!echo "Val Fight count:" && ls /content/drishti/data/RWF-2000/val/Fight/ | wc -l
!echo "Val NonFight count:" && ls /content/drishti/data/RWF-2000/val/NonFight/ | wc -l

Extracting with error handling...
Done. Skipped 24 files.

Folder structure:
train  val
Fight  NonFight
Train Fight count:
776
Train NonFight count:
800
Val Fight count:
200
Val NonFight count:
200


In [11]:
import os

# Rename to match what trainer expects
if os.path.exists('/content/drishti/data/RWF-2000'):
    os.rename('/content/drishti/data/RWF-2000',
              '/content/drishti/data/rwf2000')
    print("Renamed to rwf2000")

# Verify
!ls /content/drishti/data/rwf2000/
!ls /content/drishti/data/rwf2000/train/
print("Ready for training")

Renamed to rwf2000
train  val
Fight  NonFight
Ready for training


In [12]:
# Verify GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
!nvidia-smi


True
Tesla T4
Sat Oct  3 21:01:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P0             31W /   70W |     487MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------

In [13]:
# Train
import sys
sys.path.insert(0, '/content/drishti/src')
from drishti.actions.trainer import VideoMAETrainer

trainer = VideoMAETrainer(
    data_root='data/rwf2000',
    output_dir='models/videomae_fight',
    device='cuda',
    batch_size=4,
    num_epochs=5
)
trainer.train()


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED | 
encoder_to_decoder.weight                                       

RWF-2000 train: 776 Fight, 800 NonFight

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


RWF-2000 val: 200 Fight, 200 NonFight

...

Epoch 5/5 | train loss 0.0101 acc 0.9994 | val loss 0.3916 acc 0.8875 f1 0.8874

precision    recall  f1-score   support

    NonFight       0.87      0.92      0.89       200
       Fight       0.91      0.86      0.88       200

    accuracy                           0.89       400
   macro avg       0.89      0.89      0.89       400
weighted avg       0.89      0.89      0.89       400

Saved best_model.pt (val F1 0.8874)

Done in 62.5 min. Best val F1 0.8874 at epoch 5.

{'best_val_f1': 0.8874148574859737,
 'best_epoch': 5,
 'history': [{'epoch': 1,
   'train_loss': 0.6323227028026799,
   'train_acc': 0.6389593908629442,
   'val_loss': 0.5835570036619901,
   'val_acc': 0.625,
   'val_f1': 0.6242390841453944,
   'confusion_matrix': [[134, 66], [84, 116]],
   'classification_report': {'NonFight': {'precision': 0.6146788990825688,
     'recall': 0.67,
     'f1-score': 0.6411483253588517,
     'support': 200.0},
    'Fight': {'precision': 0.6373626373626373,
     'recall': 0.58,
     'f1-score': 0.6073298429319371,
     'support': 200.0},
    'accuracy': 0.625,
    'macro avg': {'precision': 0.6260207682226031,
     'recall': 0.625,
     'f1-score': 0.6242390841453944,
     'support': 400.0},
    'weighted avg': {'precision': 0.626020768222603,
     'recall': 0.625,
     'f1-score': 0.6242390841453944,
     'support': 400.0}}},
  {'epoch': 2,
   'train_loss': 0.4971686491203792,
   'train_acc': 0.748730964467005,
   'val_loss': 0.33315330373588947,
   'val

In [15]:
# Push to HuggingFace Hub
# Store a write token in Colab Secrets (key icon in the left sidebar) as HF_TOKEN, and enable
# notebook access. Do not paste the token into this notebook: it gets committed to GitHub.
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path='models/videomae_fight',
    repo_id='Parrv/drishti-fight-detector',
    repo_type='model',
    token=userdata.get('HF_TOKEN')
)
print("Pushed to HuggingFace Hub")


Pushed to HuggingFace Hub


In [16]:
# Quick eval
trainer.evaluate('models/videomae_fight/best_model.pt')


RWF-2000 val: 200 Fight, 200 NonFight

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Val loss 0.3916  acc 0.8875  macro-F1 0.8874

precision    recall  f1-score   support

    NonFight       0.87      0.92      0.89       200
       Fight       0.91      0.86      0.88       200

    accuracy                           0.89       400
   macro avg       0.89      0.89      0.89       400
weighted avg       0.89      0.89      0.89       400

Confusion matrix (rows = true, cols = predicted; order NonFight, Fight):

[[183  17]
 [ 28 172]]

{'loss': 0.3915549512540747,
 'acc': 0.8875,
 'f1': 0.8874148574859737,
 'report': '              precision    recall  f1-score   support\n\n    NonFight       0.87      0.92      0.89       200\n       Fight       0.91      0.86      0.88       200\n\n    accuracy                           0.89       400\n   macro avg       0.89      0.89      0.89       400\nweighted avg       0.89      0.89      0.89       400\n',
 'report_dict': {'NonFight': {'precision': 0.8672985781990521,
   'recall': 0.915,
   'f1-score': 0.8905109489051095,
   'support': 200.0},
  'Fight': {'precision': 0.91005291005291,
   'recall': 0.86,
   'f1-score': 0.884318766066838,
   'support': 200.0},
  'accuracy': 0.8875,
  'macro avg': {'precision': 0.888675744125981,
   'recall': 0.8875,
   'f1-score': 0.8874148574859737,
   'support': 400.0},
  'weighted avg': {'precision': 0.888675744125981,
   'recall': 0.8875,
   'f1-score': 0.8874148574859737,
   'support': 400.0}},
 'confusion_matrix': array([[183,  17],
   